In [1]:
from dotenv import load_dotenv
import anthropic, json, pandas as pd
from pathlib import Path
import os

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [4]:
df = pd.read_csv(root / "csv_enunciados/enunciados_ejercicios_clean.csv", encoding="utf-8")

In [5]:
load_dotenv()
client = anthropic.Anthropic()  # lee ANTHROPIC_API_KEY del entorno

In [ ]:
# system_md = (root / "classifier/matrices.md").read_text(encoding="utf-8")
# #df = pd.read_csv("../classifier/sample_classification.csv")
# df = matrices

In [6]:
def classify(enunciado, system_md, modelo="claude-haiku-4-5-20251001"):
    resp = client.messages.create(
        model=modelo,
        max_tokens=300,
        system=[{"type": "text", "text": system_md,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": f"Enunciado:\n{enunciado}"}],
    )
    texto = resp.content[0].text.strip().removeprefix("```json").removesuffix("```").strip()
    return json.loads(texto)

In [7]:
output_file = root / "classifier/resultados.jsonl"
already_processed = {json.loads(l)["id"] for l in output_file.open()} if output_file.exists() else set()

In [31]:
for index, row in df.iterrows():
    if row.id in already_processed:
        continue
    try:
        r = classify(row.enunciado, system_md)
        r["id"] = row.id
    except Exception as e:
        r = {"id": row.id, "error": str(e)}
    with output_file.open("a", encoding="utf-8") as f:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [40]:
df = pd.read_json(root / "classifier/resultados.jsonl", lines = True)

In [ ]:
df.head()

,asignatura,tema,tipo_ejercicio,confianza,id,error
0,Física,Campo gravitatorio,plano_inclinado,alta,2022_Junio_A1,NaN
1,Física,Campo gravitatorio,plano_inclinado,alta,2024_Julio_A2,NaN
2,Física,Campo gravitatorio,satélites,alta,2024_Reserva 2_A2,NaN
3,Física,Campo gravitatorio,distribución_masas,alta,2023_Junio_A2,NaN
4,Física,Campo gravitatorio,satélites,alta,2016_Reserva 2_1_A,NaN
...,...,...,...,...,...,...
299,Mates II,Matrices y determinantes,potencias,alta,mates_ii_matrices_y_determinantes_2025_reserva...,NaN
300,Mates II,Matrices y determinantes,ecuación matricial,alta,mates_ii_matrices_y_determinantes_2025_reserva...,NaN
301,Mates CCSS,Matrices y determinantes,determinantes,media,mates_ii_matrices_y_determinantes_2025_reserva...,NaN
302,Mates CCSS,Matrices y determinantes,potencias|ecuación matricial,media,mates_ii_matrices_y_determinantes_2025_reserva...,NaN


In [48]:
df.to_csv(root / "classifier/classified_exercises.csv", index = False, encoding="utf-8")

In [50]:
df.head()

,asignatura,tema,tipo_ejercicio,confianza,id,error
0,Física,Campo gravitatorio,plano_inclinado,alta,2022_Junio_A1,NaN
1,Física,Campo gravitatorio,plano_inclinado,alta,2024_Julio_A2,NaN
2,Física,Campo gravitatorio,satélites,alta,2024_Reserva 2_A2,NaN
3,Física,Campo gravitatorio,distribución_masas,alta,2023_Junio_A2,NaN
4,Física,Campo gravitatorio,satélites,alta,2016_Reserva 2_1_A,NaN


Cargar enunciados

In [11]:
df_enun = pd.read_csv(root / "csv_enunciados/enunciados_ejercicios_clean.csv",
                      encoding="utf-8")

In [12]:
df_enun.head(2)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
0,fisica_campo_electrico_y_magnetico_2025_julio_...,Física,Física,2025,Julio,Ejercicio A,Campo eléctrico y magnético,FISICA. 2025. JULIO. EJERCICIO a,Discuta la veracidad de las siguientes afirmac...,enunciados_ejercicios/Física/campo_eléctrico...,enunciados_ejercicios/Física/campo_eléctrico...
1,fisica_campo_electrico_y_magnetico_2025_julio_...,Física,Física,2025,Julio,Ejercicio B1,Campo eléctrico y magnético,FISICA. 2025. JULIO. EJERCICIO b1,Dos conductores rectilíneos muy largos se disp...,enunciados_ejercicios/Física/campo_eléctrico...,enunciados_ejercicios/Física/campo_eléctrico...


Combinar enunciados con clasificaciones

In [ ]:
merged_df = pd.merge(df, df_enun, on=["id", "asignatura"])

Exportar enunciados clasificados

In [ ]:
merged_df.to_csv(root / "csv_enunciados/enunciados_clasificados.csv",
                 index = False, encoding="utf-8")